In [1]:
import pandas as pd
import numpy as np

In [2]:
# completions_export = pd.read_csv("..\data\completions_export.csv")
completions_cip4_agg = pd.read_csv("..\data\completions_cip4_agg.csv")
scorecard_tn_export = pd.read_csv("..\data\scorecard_tn_export.csv")

In [3]:
# Create a list of unitids for each dataset
scorecard_unitids = set(scorecard_tn_export['unitid_n'])
completions_unitids = set(completions_cip4_agg['unitid_n'])

In [4]:
# Define match, scorecard_only, and completions_only
matching_unitids = scorecard_unitids & completions_unitids
scorecard_only = scorecard_unitids - completions_unitids
completions_only = completions_unitids - scorecard_unitids

In [5]:
# Create a df for the cross filter
unitid_cross = pd.DataFrame({
    'unitid_n' : list(scorecard_unitids | completions_unitids)
})

In [6]:
# Create column with .isin data
unitid_cross['in_scorecard'] = unitid_cross['unitid_n'].isin(scorecard_unitids)
unitid_cross['in_completions'] = unitid_cross['unitid_n'].isin(completions_unitids)
unitid_cross['match'] = unitid_cross['in_scorecard'] & unitid_cross['in_completions']

In [7]:
# Sort matches
unitid_cross = unitid_cross.sort_values(['match', 'unitid_n'], ascending=[False, True])
unitid_cross.head(20)                                

,unitid_n,in_scorecard,in_completions,match
3236,219505,True,True,True
3263,219587,True,True,True
3267,219596,True,True,True
3273,219602,True,True,True
3288,219639,True,True,True
3320,219709,True,True,True
3324,219718,True,True,True
3351,219790,True,True,True
3358,219806,True,True,True
3368,219824,True,True,True


In [8]:
# Examine False results
matching_unitids_df = unitid_cross[unitid_cross['match'] == False]
matching_unitids_df.head(20)

,unitid_n,in_scorecard,in_completions,match
1115,100636,False,True,False
1127,100654,False,True,False
1129,100663,False,True,False
1138,100690,False,True,False
1148,100706,False,True,False
1160,100724,False,True,False
1177,100751,False,True,False
1183,100760,False,True,False
1211,100812,False,True,False
1217,100830,False,True,False


In [9]:
# How many unitids are in both?
unitid_cross['match'].value_counts()

match
False    7722
True      129
Name: count, dtype: int64

In [10]:
# How many unitids are in TN?
scorecard_tn_export['unitid_n'].nunique()

132

In [11]:
# Test composite_keys to ensure the overlap is as expected.
scorecard_keys = set(scorecard_tn_export['composite_key'])
completions_keys = set(completions_cip4_agg['composite_key'])

In [12]:
matching_keys = scorecard_keys & completions_keys
print("Number of composite keys in both tables:", len(matching_keys))

Number of composite keys in both tables: 586


In [13]:
scorecard_only_keys = scorecard_keys - completions_keys
print("Composite keys in Scorecard but NOT in Completions:", len(scorecard_only_keys))

Composite keys in Scorecard but NOT in Completions: 2327


In [14]:
completions_only_keys = completions_keys - scorecard_keys
print("Composite keys in Completions but NOT in Scorecard:", len(completions_only_keys))

Composite keys in Completions but NOT in Scorecard: 137834


In [15]:
# What are the 3 unitids that are in Scorecard but not in Completions
scorecard_unitids = set(scorecard_tn_export['unitid_n'])
completions_unitids = set(completions_cip4_agg['unitid_n'])
missing_unitids = scorecard_unitids - completions_unitids
print("Unitids in Scorecard but not in Completions:", missing_unitids)

Unitids in Scorecard but not in Completions: {490470, 497198, 483887}


In [16]:
scorecard_tn_export[scorecard_tn_export['unitid_n'].isin(missing_unitids)][['unitid_n', 'inst_name']]

,unitid_n,inst_name
2826,483887,Mind Body Institute
2893,490470,Massage Institute of Memphis
2909,497198,Academy of Allied Health Careers
2910,497198,Academy of Allied Health Careers
2911,497198,Academy of Allied Health Careers
2912,497198,Academy of Allied Health Careers


Investigate the sturctural problems causing data to be out of alignment.

In [17]:
# Select only the structural columns
scorecard_keys_df = scorecard_tn_export[['unitid_n', 'cip4', 'credlev_n']].drop_duplicates()
completions_keys_df = completions_cip4_agg[['unitid_n', 'cip4', 'credlev_n']].drop_duplicates()

In [18]:
# Create a flag for presence in the df
scorecard_keys_df['in_scorecard'] = True
completions_keys_df['in_completions'] = True

In [19]:
# Full Outer Join for diagnosis
program_cross = scorecard_keys_df.merge(
    completions_keys_df,
    on=['unitid_n', 'cip4', 'credlev_n'],
    how='outer'
)

In [20]:
# Add match flag
program_cross['match'] = (
    program_cross['in_scorecard'] & program_cross['in_completions']
)

In [21]:
program_cross['match'].value_counts()

match
False    140161
True        586
Name: count, dtype: int64

In [22]:
scorecard_only_programs = program_cross[
    (program_cross['in_scorecard'] == True) & (program_cross['in_completions'] != True)
]

completions_only_programs = program_cross[
    (program_cross['in_scorecard'] != True) & (program_cross['in_completions'] == True)
]

In [23]:
matched_programs = program_cross[program_cross['match']]
matched_programs

,unitid_n,cip4,credlev_n,in_scorecard,in_completions,match
97408,219505,3906,1,True,True,True
97427,219587,1204,1,True,True,True
97432,219596,4603,1,True,True,True
97433,219596,4701,1,True,True,True
97435,219596,4703,1,True,True,True
...,...,...,...,...,...,...
139867,493752,5108,1,True,True,True
139921,494436,4702,1,True,True,True
139923,494436,5108,1,True,True,True
139924,494436,5109,1,True,True,True


In [24]:
summary_pct = program_cross['match'].value_counts(normalize=True) * 100
summary_pct

match
False    99.58365
True      0.41635
Name: proportion, dtype: float64

Three approaches going forward:
1. Intersection - Analyze programs that exist in both data sets
2. Dual-Data - Compare production-only vs earnings-only vs overlap
3. Earnings Focused - Consider completions as contextual reference, but not directly joined

In [27]:
# Three approach breakdown
program_cross['universe'] = np.select(
    [
        (program_cross['in_scorecard'] != True) & (program_cross['in_completions'] == True),
        (program_cross['in_scorecard'] == True) & (program_cross['in_completions'] != True),
        (program_cross['in_scorecard'] != True) & (program_cross['in_completions'] == True)
    ],
    [
        'intersection',
        'scorecard_only',
        'completions_only'
    ],
    default='none'
)

program_cross['universe'].value_counts()

# Intersection are valid rows for ROI analysis

universe
intersection      137834
scorecard_only      2327
none                 586
Name: count, dtype: int64

In [29]:
# Credential breakdown
program_cross.groupby(['credlev_n', 'universe']).size().unstack(fill_value=0)

universe,intersection,none,scorecard_only
credlev_n,,,
1,126106,503,205
2,2911,11,509
3,8817,72,1613


In [30]:
# Cip4 breakdown
program_cross.groupby(['cip4', 'universe']).size().unstack(fill_value=0)

universe,intersection,none,scorecard_only
cip4,,,
99,9653,0,0
100,190,0,6
101,368,2,3
102,83,0,0
103,261,0,1
...,...,...,...
5219,269,0,3
5220,239,0,4
5221,7,0,0


In [31]:
# Institution breakdown
program_cross.groupby(['unitid_n', 'universe']).size().unstack(fill_value=0)

universe,intersection,none,scorecard_only
unitid_n,,,
100636,47,0,0
100654,40,0,0
100663,83,0,0
100690,10,0,0
100706,53,0,0
...,...,...,...
500412,2,0,0
500421,2,0,0
500430,2,0,0


In [33]:
# Which institutions will join well?
join_quality = (
    program_cross
    .groupby('unitid_n')['match']
    .mean()
    .sort_values(ascending=False)
)

join_quality.head(10)   # best aligned schools

unitid_n
222105    0.800000
221148    0.714286
221625    0.666667
221050    0.625000
219596    0.571429
221430    0.565217
221333    0.545455
220394    0.523810
221102    0.520000
221616    0.520000
Name: match, dtype: float64

In [34]:
join_quality.tail(10)   # worst aligned schools

unitid_n
500087    0.0
500342    0.0
500379    0.0
500397    0.0
500403    0.0
500412    0.0
500421    0.0
500430    0.0
500467    0.0
100830    0.0
Name: match, dtype: float64

In [ ]:
matched_programs.to_csv(
    "matched_programs.csv",
    index=False,
    encoding="utf-8"
)